In [15]:
import os
import glob
import numpy as np
import pandas as pd

# ==========================================
# 0. 核心配置与路径嗅探
# ==========================================
OUTPUT_DIR = "/NAS/shith/uplift/0420_representation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 绝对裁判：C 模型的先验路径
C_PRIOR_PATTERN = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/**/test_dist.csv"

# 待分析的演进架构模型
MODELS = {
    "V6_Res_MoE": "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v6_res_moe/run_v6_res_moe/test_dist.csv",
    "V7_Top5": "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v7_soft_top5/run_v7_soft_top5/test_dist.csv",
    "V8_S1": "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s1_t10/run_v8_s1_t10/test_dist.csv",
    "V8_S5": "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s5/run_v8_s5/test_dist.csv",
    "V10_Loss": "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v10_conflict_both/run_v10_both/test_dist.csv"
}

def sigmoid(x):
    # 物理饱和区裁剪，防止梯度爆炸
    return 1 / (1 + np.exp(-np.clip(x, -15, 15)))

# ==========================================
# 1. 上帝视角：因果象限全量打标 (裁判)
# ==========================================
print("📥 [1/4] Tagging cohorts using C-Prior as referee...")
c_path = glob.glob(C_PRIOR_PATTERN, recursive=True)[0]
df_c = pd.read_csv(c_path)

p_always = df_c['y0_prob'] 
p_comp = np.clip(df_c['y1_prob'] - df_c['y0_prob'], 0, None)
p_never = np.clip(1.0 - df_c['y1_prob'], 0, None) 

th_always, th_comp, th_never = p_always.quantile(0.9), p_comp.quantile(0.9), p_never.quantile(0.9)
cohorts = pd.Series('Others', index=df_c.index)
cohorts.loc[(p_never >= th_never) & (p_always < th_always) & (p_comp < th_comp)] = 'NT (Tail)'
cohorts.loc[(p_comp >= th_comp) & (p_always < th_always)] = 'Pure CO (Gold)'
cohorts.loc[(p_always >= th_always) & (p_comp < th_comp)] = 'Pure AT (Wool)'
cohorts.loc[(p_always >= th_always) & (p_comp >= th_comp)] = 'AT & CO (Both)'

df_c['Base_Cohort'] = cohorts
target_groups = ['Pure CO (Gold)', 'AT & CO (Both)', 'Pure AT (Wool)', 'NT (Tail)']

# ==========================================
# 2. 核心逻辑：计算概率推力 (Physical Push)
# ==========================================
def run_push_analysis(model_name, csv_path, cohort_tags):
    print(f"🚀 Processing Model: {model_name}...")
    df = pd.read_csv(csv_path)
    df['Cohort'] = cohort_tags
    
    # --- 物理量 A: 基座状态 (Base Tower Only) ---
    # 计算 Base 塔在 Logit 空间输出的初始概率
    p0_base = sigmoid(df['wb_base_y0'])
    p1_base = sigmoid(df['wb_base_y1'])
    u_base = p1_base - p0_base
    
    # --- 物理量 B: 最终状态 (Final Output) ---
    # 这已经是 sigmoid 后的最终概率了
    p0_final = df['y0_prob']
    p1_final = df['y1_prob']
    u_final = p1_final - p0_final
    
    # --- 物理量 C: 专家推力 (Physical Push) ---
    # 这是专家介入后，真实把概率拽动了多少个百分点
    push_p0 = p0_final - p0_base # 专家对 Y0 的拉扯
    push_p1 = p1_final - p1_base # 专家对 Y1 的拉扯
    # 专家对最终因果增量(Uplift)的“净物理纠偏”
    net_correction = push_p1 - push_p0 
    
    results = []
    for grp in target_groups:
        mask = df['Cohort'] == grp
        if mask.sum() == 0: continue
        
        results.append({
            "Model": model_name,
            "Cohort": grp,
            "Base_U": u_base[mask].mean(),
            "Push_P0": push_p0[mask].mean(),
            "Push_P1": push_p1[mask].mean(),
            "Net_Corr": net_correction[mask].mean(),
            "Final_U": u_final[mask].mean()
        })
    return results

# ==========================================
# 3. 遍历模型并导出数据
# ==========================================
all_analysis_data = []
for name, path in MODELS.items():
    if os.path.exists(path):
        all_analysis_data.extend(run_push_analysis(name, path, df_c['Base_Cohort']))

report_df = pd.DataFrame(all_analysis_data)

# --- 打印对账单 ---
print("\n" + "="*110)
print(f"{'Model':<15} | {'Cohort':<15} | {'Base_U':>10} | {'Push_P0':>10} | {'Push_P1':>10} | {'Net_Corr':>10} | {'Final_U':>10}")
print("-" * 110)

for cohort in target_groups:
    sub = report_df[report_df['Cohort'] == cohort]
    for _, row in sub.iterrows():
        print(f"{row['Model']:<15} | {row['Cohort']:<15} | {row['Base_U']:>10.4f} | {row['Push_P0']:>10.4f} | {row['Push_P1']:>10.4f} | {row['Net_Corr']:>10.4f} | {row['Final_U']:>10.4f}")
    print("-" * 110)

# 保存 CSV，方便后续画图或塞入论文表格
save_path = os.path.join(OUTPUT_DIR, "probability_push_report.csv")
report_df.to_csv(save_path, index=False)
print(f"✅ [Done] 物理推力对账单已生成: {save_path}")

📥 [1/4] Tagging cohorts using C-Prior as referee...
🚀 Processing Model: V6_Res_MoE...
🚀 Processing Model: V7_Top5...
🚀 Processing Model: V8_S1...
🚀 Processing Model: V8_S5...
🚀 Processing Model: V10_Loss...

Model           | Cohort          |     Base_U |    Push_P0 |    Push_P1 |   Net_Corr |    Final_U
--------------------------------------------------------------------------------------------------------------
V6_Res_MoE      | Pure CO (Gold)  |    -0.0037 |    -0.0144 |    -0.0104 |     0.0040 |     0.0002
V7_Top5         | Pure CO (Gold)  |     0.0004 |     0.0010 |     0.0011 |     0.0001 |     0.0005
V8_S1           | Pure CO (Gold)  |     0.0252 |    -0.0024 |    -0.0250 |    -0.0226 |     0.0026
V8_S5           | Pure CO (Gold)  |     0.0045 |     0.0000 |    -0.0008 |    -0.0009 |     0.0036
V10_Loss        | Pure CO (Gold)  |     0.0151 |    -0.0028 |    -0.0153 |    -0.0125 |     0.0027
---------------------------------------------------------------------------------------